### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Importaciones

In [12]:
using CSV, DataFrames, Glob, Statistics

# Preparación de los datos (20%)

## Carga y unificación de los datos

In [5]:
base = "Datos Práctica"

# CSV del Investigador A
csv_inv_a = glob("Investigador A/day */*.csv", base)

# CSV del Investigador B
csv_inv_b = glob("Investigador B/*.csv", base)

all_csv = vcat(csv_inv_a, csv_inv_b)

dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

println("Dataset correctamente cargado y unificado.")
println("Número de variables:         ", ncol(df_total))
println("Número de instancias:        ", nrow(df_total))
println("Número de individuos:        ", length(unique(df_total.subject)))
println("Número de clases de salida:  ", length(unique(df_total.Activity)))

Dataset correctamente cargado y unificado.
Número de variables:         563
Número de instancias:        10299
Número de individuos:        30
Número de clases de salida:  6


In [6]:
println("Clases:\n")
for activity in unique(df_total.Activity)
    println(activity)
end

Clases:

STANDING


# Análisis de valores ausentes

In [10]:
# Porcentajes de nulos por variable
nulos_por_variable = DataFrame(
    Variable = names(df_total),
    PorcentajeNulos = [count(ismissing, df_total[!, col]) / nrow(df_total) * 100 for col in names(df_total)]
)

# Porcentaje de nulos en todo el dataset
total_nulos = sum(count(ismissing, df_total[!, col]) for col in names(df_total))
total_valores = nrow(df_total) * ncol(df_total)

porcentaje_total_nulos = (total_nulos / total_valores) * 100

println("Porcentaje total de valores nulos en el dataset: $(porcentaje_total_nulos)%")

Porcentaje total de valores nulos en el dataset: 0.9984242033534787%


# Tratamiento y transformación de datos

In [9]:
# Eliminamos la columna subject para entrenar
df_model = select(df_total, Not(:subject));

In [14]:
# Rellenamos con la mediana porque es menos sensible a outliers
df_imputado = deepcopy(df_total)
gdf = groupby(df_imputado, :subject) # Agrupamos por individuo para que cada dato nulo se rellene con la mediana de los valores de dicho individuo.

for subdf in gdf
    for col in names(subdf)
        if col in (:subject, :activity)
            continue
        end

        # Ignoramos variables no numéricas
        if !(eltype(subdf[!, col]) <: Number)
            continue
        end

        mediana = median(skipmissing(subdf[!, col]))
        replace!(subdf[!, col], missing => mediana)
    end
end